## 0. Введение

Данный ноутбук представляет собой скрипт для полной проверки работы системы `manuscript-ocr` версии 0.1.12 и выше от начала до конца. В примере выполняется полный цикл обработки изображения, включая детекцию, распознавание и получение итогового текста, а также проводится замер скорости работы пайплайна на CPU и GPU-устройствах.

Ноутбук предназначен для сквозной апробации библиотеки и позволяет оценить как корректность работы всех этапов обработки, так и производительность системы в различных вычислительных конфигурациях.


Минимальные технические требования для запуска примера:

- Python 3.8+
- не менее 8 ГБ оперативной памяти
- запуск на CPU поддерживается
- для запуска на GPU рекомендуется NVIDIA GPU с объемом памяти от 4 ГБ
- при использовании GPU в системе должны быть установлены совместимые драйверы NVIDIA, а также CUDA/cuDNN-окружение, соответствующее версии `onnxruntime-gpu`

Пример не требует установки тяжелых зависимостей, таких как `torch`, и может использоваться как для CPU-, так и для GPU-апробации OCR-пайплайна библиотеки.

In [1]:
!pip install "git+https://github.com/konstantinkozhin/manuscript-ocr.git"

  Cloning https://github.com/konstantinkozhin/manuscript-ocr.git to /tmp/pip-req-build-akgnwgwb
  Running command git clone --filter=blob:none --quiet https://github.com/konstantinkozhin/manuscript-ocr.git /tmp/pip-req-build-akgnwgwb
  Resolved https://github.com/konstantinkozhin/manuscript-ocr.git to commit a10edbd6a6fbfbda7ca1fad40c83cb80e9b0e8d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 99.9 MB/s eta 0:00:00
  Created wheel for manuscript-ocr: filename=manuscript_ocr-0.1.12-py3-none-any.whl size=162535 sha256=b7d28d807a32df8d80043a7890c91e5134fd8b4f2778bf9f19e0d531e43fb3e1
  Stored in directory: /tmp/pip-ephem-wheel-cache-gr4srz99/wheels/90/dd/bc/2bb9fb330d06249089ad1e27d6c90eb3376bdcd0bf91086a42
Successfully built manuscript-ocr


## 1. Установка зависимостей

In [ ]:
%cd ..
%pip install .

c:\Users\USER\manuscript-ocr-3
Obtaining file:///C:/Users/USER/manuscript-ocr-3
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached onnxruntime-1.24.4-cp311-cp311-win_amd64.whl.metadata (5.4 kB)
  Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached shapely-2.1.2-cp311-cp311-win_amd64.whl.metadata (7.1 kB)
  Using cached scikit_image-0.26.0-cp311-cp311-wi


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#может потребовать перезапуска среды в Google Colab
!pip install "manuscript-ocr>=0.1.12"

  Using cached manuscript_ocr-0.1.10-py3-none-any.whl.metadata (6.9 kB)
  Using cached onnxruntime-1.24.4-cp311-cp311-win_amd64.whl.metadata (5.4 kB)
  Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached shapely-2.1.2-cp311-cp311-win_amd64.whl.metadata (7.1 kB)
  Using cached scikit_image-0.26.0-cp311-cp311-win_amd64.whl.metadata (15 kB)
  Using cached numba-0.65.0-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached gdown-5.2.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cac


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Скачивание данных

In [8]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

from tqdm.auto import tqdm

base_url = "https://huggingface.co/datasets/anna4uonline/YeniseiGovReports-TD/resolve/main"
dataset_root = Path("datasets") / "YeniseiGovReports-TD"
dataset_root.mkdir(parents=True, exist_ok=True)

files = {
    "train_annotations": "train.json",
    "val_annotations": "test.json",
    "train_images_zip": "train_images.zip",
    "val_images_zip": "test_images.zip",
}

def download_with_progress(url: str, out_path: Path):
    if out_path.exists():
        print(f"Already exists: {out_path}")
        return

    last = [0]
    with tqdm(
        desc=f"Downloading {out_path.name}",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        def reporthook(block_num, block_size, total_size):
            if total_size > 0:
                pbar.total = total_size
            downloaded = block_num * block_size
            pbar.update(downloaded - last[0])
            last[0] = downloaded

        urlretrieve(url, out_path, reporthook=reporthook)

def extract_zip_once(zip_path: Path, extract_to: Path):
    marker = extract_to / f".{zip_path.stem}.done"
    if marker.exists():
        print(f"Already extracted: {zip_path.name}")
        return

    print(f"Extracting {zip_path.name} ...")
    with ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    marker.touch()

def resolve_images_dir(root: Path, split_name: str) -> Path:
    direct = root / f"{split_name}_images"
    nested = direct / f"{split_name}_images"
    if nested.exists():
        return nested
    if direct.exists():
        return direct
    raise FileNotFoundError(f"Could not find extracted folder for {split_name}_images")

# Скачивание файлов датасета
for _, filename in files.items():
    download_with_progress(
        f"{base_url}/{filename}",
        dataset_root / filename,
    )

# Распаковка архивов с изображениями
extract_zip_once(dataset_root / "train_images.zip", dataset_root)
extract_zip_once(dataset_root / "test_images.zip", dataset_root)

# Пути для EAST.train(...)
train_images = str(resolve_images_dir(dataset_root, "train"))
train_annotations = str(dataset_root / "train.json")

# В датасете нет отдельного val split, поэтому для валидации используем test split
val_images = str(resolve_images_dir(dataset_root, "test"))
val_annotations = str(dataset_root / "test.json")

print("train_images      =", train_images)
print("train_annotations =", train_annotations)
print("val_images        =", val_images)
print("val_annotations   =", val_annotations)


c:\Users\USER\manuscript-ocr-3\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Extracting train_images.zip ...
Extracting test_images.zip ...
train_images      = datasets\YeniseiGovReports-TD\train_images
train_annotations = datasets\YeniseiGovReports-TD\train.json
val_images        = datasets\YeniseiGovReports-TD\test_images
val_annotations   = datasets\YeniseiGovReports-TD\test.json


По умолчанию библиотека устанавливается в режиме работы на CPU устройстве.

# 3. Пример расчета времени инференса на CPU

In [3]:
import os
import time
import numpy as np
from manuscript import Pipeline
from manuscript.correctors import CharLM
from manuscript.recognizers import TRBA
from manuscript.detectors import EAST

# Папка с распакованными изображениями
extract_dir = val_images

# Расширения изображений
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif")

# Инициализация детектора (указанные модели скачаются автоматически при первом использовании)
pipeline = Pipeline(
    detector=EAST("east_50_g1", device="cpu"),  # явная передача параметра device для детектора
    recognizer=TRBA("trba_lite_g1", device="cpu"),  # явная передача параметра device для распознавателя
    corrector=CharLM("prereform_charlm_g1", device="cpu"),  # явная передача параметра device для корректора
)

# Собираем только первые 30 изображений
image_paths = []
for root, _, files in os.walk(extract_dir):
    for fname in files:
        if fname.lower().endswith(IMAGE_EXTS):
            image_paths.append(os.path.join(root, fname))

image_paths = sorted(image_paths)[:30]

# Инференс по первым 30 изображениям и замер времени
times = []
for img_path in image_paths:
    print(img_path)
    start = time.time()
    result = pipeline.predict(img_path)
    times.append(time.time() - start)

# Статистика времени
times = np.array(times)
print(f"Обработано изображений: {len(times)}")
print(f"Мин.: {times.min():.3f} сек, Макс.: {times.max():.3f} сек, Ср.: {times.mean():.3f} сек")

east_50_g1.onnx: 100%|███████████████████████| 107M/107M [00:01<00:00, 88.3MB/s]


 Downloaded to /root/.manuscript/weights/east_50_g1.onnx


trba_lite_g1.onnx: 100%|███████████████████| 36.1M/36.1M [00:00<00:00, 68.7MB/s]


 Downloaded to /root/.manuscript/weights/trba_lite_g1.onnx


trba_lite_g1.json: 100%|███████████████████| 2.80k/2.80k [00:00<00:00, 12.8kB/s]


...

 Downloaded to /root/.manuscript/weights/prereform_charlm_g1.onnx


prereform_charlm_g1.json: 100%|████████████████| 545/545 [00:00<00:00, 1.45kB/s]


 Downloaded to /root/.manuscript/weights/prereform_charlm_g1.json


prereform_words.txt: 100%|█████████████████| 34.6M/34.6M [00:00<00:00, 42.5MB/s]


 Downloaded to /root/.manuscript/weights/prereform_words.txt
datasets/YeniseiGovReports-TD/test_images/1017.jpg
[EAST] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
[TRBA] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
[CharLM] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
datasets/YeniseiGovReports-TD/test_images/104.jpg
datasets/YeniseiGovReports-TD/test_images/1052.jpg
datasets/YeniseiGovReports-TD/test_images/1054.jpg
datasets/YeniseiGovReports-TD/test_images/1055.jpg
datasets/YeniseiGovReports-TD/test_images/1061.jpg
datasets/YeniseiGovReports-TD/test_images/1062.jpg
datasets/YeniseiGovReports-TD/test_ima

'Running on: CPUExecutionProvider' свидетельствует о том, что запуск выполнился на CPU устройстве.
На процессоре `Intel(R) Core(TM) i9-14900KF` среднее время полного инференса составляет `4.128` секунды на одно изображение.

Между тем, на более слабых CPU-конфигурациях Google Colab среднее время полного инференса составляет `25.495` секунд на изображение, что также можно считать приемлемым показателем для обработки на центральном процессоре т.е. без графического ускорителя.

Стоит отметить, что время работы OCR-пайплайна напрямую зависит от количества слов на изображении: при увеличении числа текстовых фрагментов возрастает и общее время обработки.

## 3. Инференс с визуализацией результата на CPU

Теперь выполним полный OCR (детекция + распознавание) одного изображения и визуализируем результат.

In [4]:
from pathlib import Path
from urllib.request import urlretrieve

from manuscript import Pipeline
from manuscript.utils.visualization import visualize_page
from manuscript.detectors import EAST
from manuscript.recognizers import TRBA
from manuscript.correctors import CharLM

# Скачиваем пример изображения из основного репозитория
image_url = "https://raw.githubusercontent.com/konstantinkozhin/manuscript-ocr/main/example/images/img2.jpeg"
image_path = Path("img2.jpeg")

if not image_path.exists():
    urlretrieve(image_url, image_path)

# Создание OCR-пайплайна
pipeline = Pipeline(
    detector = EAST("east_50_g1", device="cpu"), # явная передача параметра device для детектора
    recognizer = TRBA("trba_lite_g1", device="cpu"), # явная передача параметра device для распознавателя
    corrector = CharLM("prereform_charlm_g1", device="cpu"), # явная передача параметра device для корректора
)

# Обработка изображения и получение результата
result = pipeline.predict(str(image_path))

print("Результат:")
print(result)
print("\n" + "=" * 50 + "\n")

# Извлечение распознанного текста
text = pipeline.get_text(result["page"])
print("Распознанный текст:")
print(text)
print("\n" + "=" * 50 + "\n")

# Визуализация результата
vis_img = visualize_page(str(image_path), result["page"])
vis_img.show()  # или vis_img.save("result.jpg") для сохранения

[EAST] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
[TRBA] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
[CharLM] Device configuration:
  Requested device: cpu
  Requested providers: ['CPUExecutionProvider']
  Active providers: ['CPUExecutionProvider']
  Running on: CPUExecutionProvider
Результат:
{'page': Page(blocks=[Block(lines=[Line(text_spans=[TextSpan(polygon=[(1084.718505859375, 156.11898803710938), (1123.6890869140625, 156.11898803710938), (1123.6890869140625, 190.14610290527344), (1084.718505859375, 190.14610290527344)], detection_confidence=0.9963154196739197, text='11', recognition_confidence=0.9999991059303284, order=0)], order=0), Line(text_spans=[TextSpan(polygon=[(223.2061309814453, 252.7427520751953), (323.67056274414

## 4. Переустановка под GPU

Согласно документации https://konstantinkozhin.github.io/manuscript-ocr/ для установки под GPU-устройство необходимо удалить зависимость `onnxruntime` и установить версию `onnxruntime-gpu`. При этом для работы с GPU-устройством в системе должны быть установлены совместимые драйверы NVIDIA, а также библиотеки CUDA и cuDNN, соответствующие версии `onnxruntime-gpu`. Переустанавливать саму библиотеку `manuscript-ocr` при этом не требуется.

In [3]:
# Безопасная переустановка ONNX Runtime для GPU
!pip uninstall -y onnxruntime
!pip install --no-cache-dir --force-reinstall onnxruntime-gpu==1.24.4
!pip install --no-cache-dir nvidia-cudnn-cu12 nvidia-cublas-cu12 nvidia-cuda-runtime-cu12 nvidia-cufft-cu12

Found existing installation: onnxruntime 1.24.4
Uninstalling onnxruntime-1.24.4:
  Successfully uninstalled onnxruntime-1.24.4


   ---------------------------------------- 0.0/207.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/207.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/207.2 MB 660.6 kB/s eta 0:05:14
   ---------------------------------------- 0.1/207.2 MB 469.7 kB/s eta 0:07:21
   ---------------------------------------- 0.1/207.2 MB 525.1 kB/s eta 0:06:35
   ---------------------------------------- 0.1/207.2 MB 525.1 kB/s eta 0:06:35
   ---------------------------------------- 0.2/207.2 MB 737.3 kB/s eta 0:04:41
   ---------------------------------------- 0.2/207.2 MB 808.4 kB/s eta 0:04:17
   ---------------------------------------- 0.3/207.2 MB 863.3 kB/s eta 0:04:00
   ---------------------------------------- 0.5/207.2 MB 1.2 MB/s eta 0:02:52
   ---------------------------------------- 0.5/207.2 MB 1.1 MB/s eta 0:03:05
   ---------------------------------------- 0.8/207.2 MB 1.6 MB/s eta 0:02:07
   ---------------------------------------- 1.0/207.2 MB 1.9 MB/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
manuscript-ocr 0.1.12 requires onnxruntime>=1.16, which is not installed.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/644.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/644.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/644.0 MB 435.7 kB/s eta 0:24:38
   ---------------------------------------- 0.1/644.0 MB 409.6 kB/s eta 0:26:13
   ---------------------------------------- 0.1/644.0 MB 525.1 kB/s eta 0:20:27
   ---------------------------------------- 0.1/644.0 MB 610.6 kB/s eta 0:17:35
   ---------------------------------------- 0.2/644.0 MB 860.2 kB/s eta 0:12:29
   ---------------------------------------- 0.3/644.0 MB 787.7 kB/s eta 0:13:38
   ---------------------------------------- 0.4/644.0 MB 1.2 MB/s eta 0:08:58
   ---------------------------------------- 0.5/644.0 MB 1.3 MB/s eta 0:08:11
   ---------------------------------------- 0.7/644.0 MB 1.6 MB/s eta 0:06:36
   ---------------------------------------- 1.0/644.0 MB 2.0 MB/s eta 0:05:17
   ---------------------------------------- 1.3/644.0 MB 2.3 MB/s 


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import onnxruntime as ort

print("file:", ort.__file__)
print("version:", ort.__version__)
ort.preload_dlls(directory="")
print("providers:", ort.get_available_providers())

file: c:\Users\USER\manuscript-ocr-3\env\Lib\site-packages\onnxruntime\__init__.py
version: 1.24.4
providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


На некоторых устройствах для корректной работы `onnxruntime-gpu` может потребоваться установка дополнительных runtime-зависимостей CUDA/cuDNN. Подробнее в документации: https://konstantinkozhin.github.io/manuscript-ocr/0.1.12/ru/getting_started.html

# 5. Пример расчета времени инференса на GPU

In [7]:
from pathlib import Path

dataset_root = Path("datasets") / "YeniseiGovReports-TD"

def resolve_images_dir(root: Path, split_name: str) -> Path:
    direct = root / f"{split_name}_images"
    nested = direct / f"{split_name}_images"
    if nested.exists():
        return nested
    if direct.exists():
        return direct
    raise FileNotFoundError(f"Could not find extracted folder for {split_name}_images")

train_images = str(resolve_images_dir(dataset_root, "train"))
train_annotations = str(dataset_root / "train.json")

# В датасете нет отдельного val split, поэтому для валидации используем test split
val_images = str(resolve_images_dir(dataset_root, "test"))
val_annotations = str(dataset_root / "test.json")

print("train_images      =", train_images)
print("train_annotations =", train_annotations)
print("val_images        =", val_images)
print("val_annotations   =", val_annotations)

FileNotFoundError: Could not find extracted folder for train_images

In [9]:
import os
import time
import numpy as np
from manuscript import Pipeline
from manuscript.correctors import CharLM
from manuscript.recognizers import TRBA
from manuscript.detectors import EAST

# Папка с распакованными изображениями
extract_dir = val_images

# Расширения изображений
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif")

# Инициализация детектора
pipeline = Pipeline(
    detector=EAST("east_50_g1", device="cuda"),
    # при этом можно разные компоненты пайплайна запускать на разных устройствах,
    # например, детектор и корректор на GPU, а распознаватель на CPU
    recognizer=TRBA("trba_lite_g1", device="cuda", batch_size=256),
    corrector=CharLM("prereform_charlm_g1", device="cuda"),
)

# Собираем только первые 30 изображений
image_paths = []
for root, _, files in os.walk(extract_dir):
    for fname in files:
        if fname.lower().endswith(IMAGE_EXTS):
            image_paths.append(os.path.join(root, fname))

image_paths = sorted(image_paths)[:30]

# Инференс по первым 30 изображениям и замер времени
times = []
for img_path in image_paths:
    print(img_path)
    start = time.time()
    result = pipeline.predict(img_path)
    times.append(time.time() - start)

# Статистика времени
times = np.array(times)
print(f"Обработано изображений: {len(times)}")
print(f"Мин.: {times.min():.3f} сек, Макс.: {times.max():.3f} сек, Ср.: {times.mean():.3f} сек")

ModuleNotFoundError: No module named 'manuscript'

Скорость работы на GPU зависит не только от производительности самого устройства, но и от размера батча. При небольших батчах ускорение может быть незначительным, тогда как при увеличении батча производительность GPU обычно используется эффективнее, но требует больше памяти.

 'Running on: CUDAExecutionProvider' свидетельствует о том, что запуск выполнился на GPU устройстве

## 6. Инференс с визуализацией результата на GPU

Теперь выполним полный OCR (детекция + распознавание) одного изображения и визуализируем результат.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from manuscript import Pipeline
from manuscript.utils.visualization import visualize_page
from manuscript.detectors import EAST
from manuscript.recognizers import TRBA
from manuscript.correctors import CharLM

# Скачиваем пример изображения из основного репозитория
image_url = "https://raw.githubusercontent.com/konstantinkozhin/manuscript-ocr/main/example/images/img1.jpeg"
image_path = Path("img1.jpeg")

if not image_path.exists():
    urlretrieve(image_url, image_path)

# Создание OCR-пайплайна
pipeline = Pipeline(
    detector = EAST("east_50_g1", device="cuda"), # явная передача параметра device для детектора
    # при этом можно разные компоненты пайплайна запускать на разных устройствах, например, детектор и корректор на GPU, а распознаватель на CPU
    recognizer = TRBA("trba_lite_g1", device="cpu"), # явная передача параметра device для распознавателя
    corrector = CharLM("prereform_charlm_g1", device="cuda"), # явная передача параметра device для корректора
)

# Обработка изображения и получение результата
result = pipeline.predict(str(image_path))

print("Результат:")
print(result)
print("\n" + "=" * 50 + "\n")

# Извлечение распознанного текста
text = pipeline.get_text(result["page"])
print("Распознанный текст:")
print(text)
print("\n" + "=" * 50 + "\n")

# Визуализация результата
vis_img = visualize_page(str(image_path), result["page"])
vis_img.show()  # или vis_img.save("result.jpg") для сохранения

Таким образом, ноутбук показывает полный цикл работы `manuscript-ocr` от начала до конца и может использоваться как для проверки корректности работы OCR-пайплайна, так и для базовой оценки скорости его выполнения на CPU и GPU. Дополнительным преимуществом является то, что для такого сценария не требуется установка тяжелых зависимостей, например `torch`. Для перехода на GPU-исполнение достаточно заменить `onnxruntime` на `onnxruntime-gpu` при условии, что в системе уже установлено и корректно настроено CUDA-окружение.